# Realtime Voice and Live Multimodal Assistants

## Phase 20 - Notebook 5

**What you will learn:**
- How realtime voice assistants differ from text streaming apps
- The transport split: SSE vs WebSocket vs WebRTC
- Turn-taking, interruption handling, and cancellation semantics
- Session state for speech-to-speech and live multimodal systems
- Audio chunking basics and latency budgeting
- Why browser, mic, camera, and tool loops need explicit state machines

**Why this matters in 2026:** modern assistants increasingly combine text, audio, vision, and tool use in a single live session. The hard part is no longer only model quality; it is managing latency, interruption, media transport, and state safely.

---

## Part 1: Architecture

A practical live assistant usually has four loops running at once:

1. **Media loop** - microphone, speaker, camera, screen, or browser events
2. **Model loop** - transcription, reasoning, response generation, tool calls
3. **Session loop** - state, memory, permissions, turn ownership
4. **Safety loop** - interruption, confirmation, moderation, sensitive actions

```text
Mic / Camera / Screen
        |
        v
  WebRTC or WebSocket transport
        |
        v
  Realtime session manager
   |        |         |
   |        |         +--> tool runner
   |        +------------> memory / context
   +---------------------> model stream

Output audio / captions / UI updates
```

**Rule of thumb:**
- Use **SSE** for one-way text streaming.
- Use **WebSocket** for app-controlled bidirectional streams.
- Use **WebRTC** when low-latency browser media is a first-class requirement.

In [1]:
from dataclasses import dataclass
from enum import Enum

class TurnState(str, Enum):
    IDLE = 'idle'
    LISTENING = 'listening'
    THINKING = 'thinking'
    SPEAKING = 'speaking'
    INTERRUPTED = 'interrupted'

@dataclass
class SessionState:
    state: TurnState = TurnState.IDLE
    current_turn: int = 0
    buffered_transcript: str = ''

    def handle(self, event: str, payload: str = ''):
        if event == 'user_started':
            self.current_turn += 1
            self.state = TurnState.LISTENING
            self.buffered_transcript = ''
        elif event == 'partial_transcript' and self.state == TurnState.LISTENING:
            self.buffered_transcript += payload
        elif event == 'user_stopped' and self.state == TurnState.LISTENING:
            self.state = TurnState.THINKING
        elif event == 'model_started' and self.state == TurnState.THINKING:
            self.state = TurnState.SPEAKING
        elif event == 'interrupt':
            self.state = TurnState.INTERRUPTED
        elif event == 'resume_listening':
            self.state = TurnState.LISTENING
        elif event == 'complete':
            self.state = TurnState.IDLE
        return {
            'turn': self.current_turn,
            'state': self.state.value,
            'transcript': self.buffered_transcript,
        }

session = SessionState()
timeline = [
    ('user_started', ''),
    ('partial_transcript', 'book a table for two '),
    ('partial_transcript', 'tomorrow night'),
    ('user_stopped', ''),
    ('model_started', ''),
    ('interrupt', ''),
    ('resume_listening', ''),
]

for event, payload in timeline:
    print(event, '->', session.handle(event, payload))

user_started -> {'turn': 1, 'state': 'listening', 'transcript': ''}
partial_transcript -> {'turn': 1, 'state': 'listening', 'transcript': 'book a table for two '}
partial_transcript -> {'turn': 1, 'state': 'listening', 'transcript': 'book a table for two tomorrow night'}
user_stopped -> {'turn': 1, 'state': 'thinking', 'transcript': 'book a table for two tomorrow night'}
model_started -> {'turn': 1, 'state': 'speaking', 'transcript': 'book a table for two tomorrow night'}
interrupt -> {'turn': 1, 'state': 'interrupted', 'transcript': 'book a table for two tomorrow night'}
resume_listening -> {'turn': 1, 'state': 'listening', 'transcript': 'book a table for two tomorrow night'}


In [2]:
import array

def chunk_pcm16(samples, sample_rate=16000, frame_ms=20):
    """Yield fixed-duration PCM16 frames."""
    frame_size = int(sample_rate * (frame_ms / 1000.0))
    for start in range(0, len(samples), frame_size):
        frame = samples[start:start + frame_size]
        if len(frame) == frame_size:
            yield frame

# Fake 100 ms mono waveform at 16 kHz
fake_wave = array.array('h', [200] * 1600)
frames = list(chunk_pcm16(fake_wave, sample_rate=16000, frame_ms=20))

print('frames:', len(frames))
print('samples per frame:', len(frames[0]))
print('frame duration ms:', 1000 * len(frames[0]) / 16000)

frames: 5
samples per frame: 320
frame duration ms: 20.0


## Part 2: Interruption Semantics

Realtime systems need explicit rules for interruption. If the user starts speaking while the model is still talking, decide all three of these quickly:

- **cancel output** - stop the assistant audio immediately
- **preserve context** - decide whether the partially spoken answer should be kept in memory
- **transfer turn ownership** - return the session to listening state

Without that policy, live assistants feel laggy, rude, or unstable.

In [3]:
class InterruptibleOutput:
    def __init__(self):
        self.tokens = []
        self.cancelled = False

    def push(self, token: str):
        if not self.cancelled:
            self.tokens.append(token)

    def cancel(self):
        self.cancelled = True

    def committed_text(self):
        return ''.join(self.tokens)

output = InterruptibleOutput()
for token in ['Sure', ', ', 'I can', ' help', ' with', ' that.']:
    output.push(token)
    if token == ' help':
        output.cancel()

print('cancelled:', output.cancelled)
print('committed text:', repr(output.committed_text()))

cancelled: True
committed text: 'Sure, I can help'


## Part 3: Current API Patterns

For new OpenAI work, prefer the Responses API and realtime transports that match your media needs. A typical text-first streaming call looks like:

```python
from openai import OpenAI

client = OpenAI()
stream = client.responses.create(
    model='gpt-4.1',
    input='Summarize the latest meeting notes.',
    stream=True,
)

for event in stream:
    if event.type == 'response.output_text.delta':
        print(event.delta, end='', flush=True)
```

For browser-native low-latency voice or live camera/screen experiences, the engineering question is usually less about Python syntax and more about session design: WebRTC transport, turn boundaries, interruption policy, tool permissions, and safe confirmation for sensitive actions.

## Project Ideas

1. Build a voice assistant that cancels speech immediately when the user interrupts.
2. Add live captions plus partial transcript display to a streaming chat UI.
3. Prototype a screen-aware copilot that separates browser media transport from tool execution.
4. Measure end-to-end latency budget across capture, transport, model TTFT, and playback.